<!-- cabecera-entorno -->
## Antes de empezar

**Clase 2 · Python y pandas: cargar, mirar, seleccionar y filtrar** — Bloque 2 · Demo. Este
notebook se recorre **por su cuenta**: explica cada concepto antes de usarlo, y el profesor circula
por el salón resolviendo dudas. No hay que esperar a que alguien lo dicte.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md). Es la única comprobación del entorno que hace este
cuaderno, y revisa todo de una vez: el intérprete, la versión de Python, las cuatro librerías con
piso de versión y el CSV de la clase.

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |
| `POR DEBAJO del piso` en alguna librería | Se instaló una versión más vieja que la verificada | Manual, secciones 3.1 y 7.2: con `(.venv)` activo, `pip install -r requirements.txt --upgrade` |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
# No instala nada: solo reporta lo que ya está montado.
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")
print("Python:", sys.version.split()[0], "- el curso pide 3.12 o superior")
print()

PISOS = {"pandas": "3.0.5", "numpy": "2.5.2", "scipy": "1.18.0", "scikit-learn": "1.9.0"}


def numeros(texto):
    partes = []
    for trozo in texto.split("."):
        digitos = "".join(c for c in trozo if c.isdigit())
        partes.append(int(digitos) if digitos else 0)
    return tuple(partes)


print("Librerías con piso de versión en requirements.txt:")
for paquete, piso in PISOS.items():
    try:
        instalada = version(paquete)
    except PackageNotFoundError:
        print(f"  {paquete:<13} NO INSTALADA  -> pip install -r requirements.txt, con (.venv) activo")
        continue
    estado = "ok" if numeros(instalada) >= numeros(piso) else f"POR DEBAJO del piso {piso}"
    print(f"  {paquete:<13} {instalada:<9} {estado}")
print()

RUTA_VERIFICACION = "../datos/estados_financieros.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 2 · Demo — Cargar, mirar, seleccionar y filtrar

**Dataset:** `../datos/estados_financieros.csv`

## Cómo se usa este notebook

Este notebook está escrito para que usted avance solo. Cada bloque de código viene precedido de la
explicación del concepto que usa, y cada término nuevo se define la primera vez que aparece. No hay
que haber programado antes: hay que leer antes de ejecutar.

**El recorrido, de lo simple a lo complejo:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 0 a 2 | Qué es un DataFrame, cómo se carga, y qué de Excel sirve aquí | El dato en memoria |
| 3 y 4 | Mirar los datos y entender sus tipos | Saber qué tiene entre manos |
| 5 y 6 | Seleccionar columnas y filas | Quedarse con un pedazo |
| 7 y 8 | La máscara booleana y sus combinaciones | El corazón de la clase |
| 9 y 10 | Los errores típicos, provocados a propósito | Saber qué hacer cuando falle |
| 11 y 12 | Atajos legibles y una pregunta de negocio real | Cerrar el círculo |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. Escribir código es el bloque 3, con el reto, y es
lo que se entrega.

**Las nueve preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.**
Abrirlo antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación es que usted sepa
mirar un resultado y decir qué significa, no que sepa reconocer una respuesta correcta cuando la
ve.

**Las cajas "Para entender qué está pasando"** van en bloque citado, con la barra vertical a la
izquierda. Explican la herramienta de Python que hay por debajo de lo que se está haciendo: qué es
una lista, qué es un parámetro con nombre, qué hace pandas con los datos que faltan. **No son
materia de analítica, son el piso para entenderla.** Si ya programó antes, sáltelas sin culpa; si
nunca lo hizo, son la diferencia entre entender y copiar. Las clases siguientes las dan por leídas.

**Si algo se rompe**, no es un accidente: la sección 9 y la 10 provocan errores a propósito, con la
explicación al lado. Ese es el material más útil del notebook.

**Punto de control:** al final hay tres preguntas para responderse a sí mismo antes de pasar al
reto.

---

## El entorno de trabajo

### Los comandos, en orden

Esta es la secuencia completa para montar el entorno de un proyecto de Python, de arriba abajo, en
la **terminal** y parado en la carpeta del curso. Es la misma para este curso, para su tesis y para
el primer día de su primer trabajo.

**Paso 1 · Crear el entorno virtual.** Una sola vez en todo el semestre.

```
python -m venv .venv          # Windows
python3 -m venv .venv         # macOS
```

**Paso 2 · Activarlo.** Cada vez que abre una terminal nueva. Salió bien si aparece `(.venv)` al
principio de la línea.

```
.venv\Scripts\Activate.ps1      # Windows (PowerShell)
source .venv/bin/activate       # macOS
```

**Paso 3 · Instalar las librerías del curso.** Una sola vez, con `(.venv)` a la vista.

```
pip install -r requirements.txt
```

**Paso 4 · Comprobar que quedó.** En la terminal, si quiere verlo desde afuera. Dentro del cuaderno
esa comprobación ya la hizo la primera celda de este archivo.

```
pip list
python -c "import pandas, numpy, matplotlib; print('Entorno listo')"
```

> Estos comandos van en la **terminal**, no en una celda: si este archivo abrió, es porque usted ya
> los corrió al seguir [`../INSTALACION.md`](../INSTALACION.md). Un notebook no puede crear el
> entorno dentro del cual él mismo se está ejecutando. Quedan escritos aquí para que sepa qué fue lo
> que tecleó, y para el día que arranque un proyecto suyo desde cero.

Lo que sigue es qué hace cada paso y por qué, que es lo que convierte esos cuatro comandos en algo
que puede repetir sin copiarlo.

### Paso 1 · Un entorno virtual es una caja de librerías por proyecto

Su computador trae un Python instalado, el **Python del sistema**. Si instalara ahí todas las
librerías, todos sus proyectos compartirían la misma caja: el de este semestre, el del semestre que
viene y lo que el propio sistema operativo use por su cuenta. El día que un proyecto necesite una
versión de pandas y otro necesite otra, uno de los dos se rompe, y desenredarlo es desagradable.

Un **entorno virtual** es una carpeta —la del curso se llama `.venv`— con una copia aislada de
Python y sus propias librerías. Todo lo que instale mientras está activo se guarda ahí adentro y en
ningún otro lado. La regla mental: **una carpeta de proyecto, un entorno virtual.** Con eso, el peor
escenario posible deja de ser grave: si algo se enreda, se borra `.venv`, se vuelve a crear y se
pierden cinco minutos, no un semestre.

### Paso 2 · Activar es cambiarle el `python` a esa terminal

Activar es decirle a esa ventana: "de aquí en adelante, cuando diga `python` o `pip`, use los de
esta carpeta". La señal de que funcionó es el prefijo `(.venv)` al principio de la línea. Por eso el
paso 1 se hace una vez y el paso 2 se repite: la carpeta `.venv` queda para siempre, la activación
no.

> **La trampa número uno del semestre.** La activación dura lo que dure esa ventana de terminal. Al
> cerrarla se pierde, y mañana hay que activar otra vez. Ese es el origen del 90% de los "ayer me
> funcionaba y hoy no": no se desinstaló nada, es que el entorno no está activo.

### Paso 3 · Qué hace `pip install`

**pip** es el instalador de librerías de Python. `pip install pandas` va a internet, descarga pandas
y lo deja **dentro del entorno virtual activo en ese momento**. Eso es todo lo que significa
"instalar una librería": traer código escrito por otra gente a un lugar donde su `import` lo pueda
encontrar.

De ahí sale el error más común de la primera semana: se instala con el entorno apagado, la librería
queda en el Python del sistema, y el notebook —que corre con el Python del entorno— no la
encuentra. El síntoma es `ModuleNotFoundError`. La causa casi nunca es que falte instalar algo, es
que se instaló del lado equivocado.

### Paso 3 · Qué hace `-r requirements.txt`, y por qué lleva pisos de versión

Instalar diez librerías a mano son diez oportunidades de escribir mal un nombre y diez versiones
distintas por salón. Por eso el paso 3 no nombra ninguna librería: el curso trae un archivo de texto
con la lista completa, `requirements.txt`, una librería por línea, y `-r` le dice a pip que la lea y
las instale todas.

Ábralo, es legible. Cuatro líneas llevan un `>=` con un número: `pandas`, `numpy`, `scipy` y
`scikit-learn`. Eso es un **piso de versión**, y la razón es concreta: son las cuatro librerías que
producen los números que usted ve en los cuadernos. Si dos estudiantes instalan en fechas distintas
y a uno le toca una versión más vieja, los resultados dejan de coincidir y el **verificador del
reto** —que compara una huella de su resultado contra la esperada— empieza a rechazar respuestas que
están bien. El piso es la versión con la que el material está verificado.

Es un piso, no un clavo: `>=` deja instalar versiones más nuevas sin problema. Lo único que no se
permite es una más vieja que la verificada.

### Paso 4 · Ya lo corrió: es la primera celda de este archivo

**Aquí no hay nada que ejecutar.** El paso 4 fue lo primero que hizo al abrir el cuaderno: la celda
de *"Antes de empezar"*, arriba del todo, **es** la comprobación, y lo que imprimió es exactamente
esto:

- **El intérprete**, la ruta del Python que está ejecutando este archivo. Tiene que llevar `.venv`.
- **La versión de Python**, que debe ser 3.12 o superior.
- **Las cuatro librerías con piso de versión** —`pandas`, `numpy`, `scipy`, `scikit-learn`— con su
  versión instalada y un `ok` al lado si cumple el piso del que habla el paso 3.
- **El CSV de hoy**, para saber si el cuaderno está parado en la carpeta correcta.

Suba a mirarla otra vez si no la leyó con cuidado. Si las cuatro librerías dicen `ok`, el intérprete
tiene `.venv` en la ruta y el CSV aparece, el entorno está bien y no hay nada más que hacer con él en
todo el semestre, salvo activarlo cada vez que abra una terminal nueva.

**Por qué esa celda y no `pip list`.** Las otras dos señales del entorno —el prefijo `(.venv)` en la
terminal y el intérprete que muestra VSCode arriba a la derecha— son útiles pero indirectas: hablan
de la terminal y del editor. La celda la responde el mismo Python que corre este archivo, que es el
único que importa. Por eso está de primera: si algo está mal, usted se entera antes de leer nada.

Si algo no cuadra, no improvise: el manual con los diez problemas frecuentes y su solución está en
[`../INSTALACION.md`](../INSTALACION.md), y casi todo lo que le puede pasar hoy ya está escrito ahí.

Con eso listo, empieza la clase.

---

## Qué estamos haciendo: EDA, análisis exploratorio de datos

Esta sección no tiene código. Está aquí para que la pueda volver a consultar en las clases 3, 4 y 5,
cuando ya no esté la diapositiva al frente.

**EDA es *Exploratory Data Analysis*, en español análisis exploratorio de datos.** Es lo primero que
se hace en cualquier análisis, y consiste en **entender los datos, limpiarlos y organizarlos,
describirlos y relacionarlos**.
Aprenda la sigla: es la que va a encontrar en la documentación, en cualquier curso en inglés y en una
oferta de trabajo.

**Y lo más importante, contra la lectura fácil: el EDA no es el trámite previo al análisis, es
análisis.** La mayor parte de los hallazgos de un analista salen aquí, explorando, y no del modelo
del final. Quien trate estas cuatro clases como preparación llega a la clase 6 sin nada que contar.

| Sí responde | No responde |
|-------------|-------------|
| ¿Qué pasó? ¿Cuánto? ¿Dónde y cuándo? | ¿Por qué pasó? |
| ¿Cuál es el valor típico y cuánto se apartan los demás? | ¿Qué va a pasar el mes que viene? |
| ¿Qué es raro aquí? ¿Qué se mueve junto con qué? | ¿Qué pasaría si cambiamos esto? |

**Los cuatro verbos, y la clase donde vive cada uno:**

| Verbo | Clase | Qué se hace | Por qué va aquí y no después |
|-------|-------|-------------|------------------------------|
| **Entender** | 2 (hoy) | Cargar, mirar, seleccionar, filtrar | Antes de opinar hay que saber qué hay en la tabla |
| **Limpiar y organizar** | 3 | Nulos, tipos, duplicados, texto, dominio | Un promedio sobre datos sucios no sale mal escrito: sale mal, y no avisa |
| **Describir** | 4 | Cada variable de a una: valor típico, dispersión y forma | Cruzar dos columnas que no se entienden por separado es adivinar con más trabajo |
| **Relacionar** | 5 | Unas variables contra otras, y el terreno listo para decidir | Solo aquí se pregunta qué se mueve junto con qué, y se aprende a no llamarlo causa |

**Dónde encaja en el proceso de la clase 1** (*de una pregunta a una decisión*): el EDA es el paso 2
(entender los datos), el paso 3 (prepararlos) y la mitad exploratoria del paso 4 (analizar). El paso
1 le entrega la pregunta y los pasos 5 y 6 vienen después.

**Lo que queda fuera, y cuándo llega.** Afirmar algo sobre lo que **no** se midió es *inferencia*
(clase 13). Decir qué va a pasar es *predicción* (clase 14). Las dos se paran encima de estas cuatro
clases: sin un EDA confiable no hay nada que inferir ni que predecir.

**Las cuatro clases se llaman EDA, y ninguna lleva numeral.** Los títulos de las clases 4 y 5 dicen
"EDA" a secas y describen su contenido (univariado, bivariado): el EDA no empieza en la clase 4,
empieza hoy.

---

## 0. El dataset, antes de tocarlo

Regla de la casa: nunca se ejecuta una línea de código sobre un dataset que no se sabe qué es.

**Qué es un CSV.** *Comma-Separated Values*: un archivo de texto plano donde cada línea es una fila
y las columnas van separadas por comas. No tiene fórmulas, ni colores, ni pestañas, ni formato. Es
el formato universal de intercambio de datos justamente por eso: cualquier programa lo lee.

**Qué es un dato tabular.** Datos organizados en filas y columnas, donde **cada fila es una
observación** (un hecho registrado) y **cada columna es una variable** (un atributo de ese hecho).
Aquí cada fila es un rubro contable de un año. Casi toda la analítica de datos ocurre sobre datos
tabulares, y pandas existe para eso.

**Origen:** datos.gov.co. Estados financieros de una entidad pública colombiana, ejercicios 2016 a
2021. Es un reporte contable: ingresos, gastos y el excedente del ejercicio, desagregado por rubro.

**Forma:** 318 filas x 3 columnas. Es pequeño a propósito, para que quepa en la cabeza.

| Columna | Qué es | Ejemplo |
|---------|--------|---------|
| `Año` | Año fiscal del reporte | 2016, 2021 |
| `CUENTA` | Rubro contable | Ingresos Fiscales, Gasto Social, Total Gastos |
| `Valor (Miles)` | Monto del rubro, en miles de pesos | 113,718,902 |

**Qué tiene de sucio (y esto importa desde el primer minuto):**

- Los números vienen con **coma como separador de miles**: `2,016` y `113,718,902`. Para pandas,
  una coma dentro de un valor significa "esto es texto". Resultado: las tres columnas se leen como
  texto y no se puede hacer aritmética con ellas.
- Hay **2 filas duplicadas** exactas.
- La columna `CUENTA` tiene 62 valores distintos, con inconsistencias de mayúsculas:
  `Devoluciones Y Descuentos` y `Devoluciones y Descuentos` son, para pandas, dos cuentas
  diferentes.

Hoy solo parchamos lo primero, porque sin números no se puede filtrar. Lo demás es la clase 3.

**Esto no es mala suerte.** Un archivo publicado por una entidad pública casi nunca llega listo
para analizar. La limpieza no es un paso previo molesto: es entre el 60% y el 80% del trabajo real
de un analista. Empezar viendo datos sucios es empezar viendo el oficio.

---

## 1. Importar pandas

**Qué es una librería.** Un paquete de código que alguien más escribió y publicó para que usted no
tenga que reinventarlo. Python trae unas cuantas de fábrica; el resto se instala (eso hizo el
`pip install pandas` del pre-work) y se **importa** en cada archivo donde se vaya a usar.

**Qué es pandas.** La librería estándar de manipulación de datos tabulares en Python. Nace en 2008
en un fondo de inversión, para hacer con código lo que se hacía a mano en Excel. Hoy es la puerta
de entrada de prácticamente todo trabajo de datos en Python.

**Qué es un alias.** `import pandas as pd` significa "importa pandas y llámalo `pd` de aquí en
adelante". El alias `pd` no es obligatorio para Python: funciona con cualquier nombre. Pero es
convención universal, y toda la documentación, todo StackOverflow y todo el código que va a leer en
su vida usan `pd`. Usar otro alias es escribir en un dialecto que nadie más lee.

In [ ]:
import pandas as pd

print("pandas:", pd.__version__)

---

## 2. Cargar el archivo: `pd.read_csv()`

**Qué es un DataFrame.** El objeto central de pandas: una tabla en memoria, con filas y columnas
nombradas. Piénselo como una hoja de cálculo que vive dentro del programa en vez de dentro de una
ventana. Tiene dos dimensiones (filas y columnas), nombres de columna, y un **índice** de fila.

**Por qué "en memoria".** `read_csv` no abre una ventana ni deja el archivo conectado: **lee el
archivo completo y lo copia a la memoria RAM** del computador, y a partir de ahí trabaja sobre esa
copia. Consecuencias que hay que tener claras desde hoy:

- Todo lo que usted haga sobre el DataFrame **no toca el archivo original**. El CSV en disco queda
  intacto pase lo que pase. Es imposible dañar los datos de origen desde el notebook.
- Si cierra el notebook, el DataFrame se pierde. Hay que volver a ejecutar `read_csv`.
- **Si el archivo es enorme, no cabe.** Un CSV de 5 GB no entra en un portátil de 8 GB de RAM, y el
  intento termina en un `MemoryError` o en el computador paginando durante minutos. Para eso
  existen tres salidas que usaremos más adelante en el semestre: `nrows=1000` para leer solo las
  primeras filas y explorar, `usecols=['col1','col2']` para leer solo las columnas que importan, y
  `chunksize=` para procesar el archivo por pedazos. Las 318 filas de hoy y las 20.000 del reto
  entran de sobra: en un portátil normal pandas maneja varios millones de filas sin quejarse.

**Sobre la ruta.** Los dos puntos del principio son la instrucción: `../` significa "suba un nivel
desde la carpeta donde está este cuaderno", y lo que sigue es la carpeta de datos y el nombre del
archivo. La ruta es relativa **al cuaderno**, no al proyecto ni a la carpeta que tenga abierta en
VSCode. Este es el error número uno del semestre, y por eso el cuaderno lo provoca de una vez, unas
celdas más abajo.

pandas tiene hermanos de esta función: `read_excel`, `read_json`, `read_sql`, `read_parquet`. Todas
devuelven lo mismo: un DataFrame. Cambia la fuente, no el objeto.

In [ ]:
df = pd.read_csv('../datos/estados_financieros.csv')

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

`df.shape` devuelve una pareja `(filas, columnas)`. `df.shape[0]` son las filas, `df.shape[1]` las
columnas. Lo va a usar todo el semestre para verificar que un filtro hizo lo que esperaba.

Sobre el nombre `df`: es la convención para "el DataFrame con el que estoy trabajando". Cuando
tenga varios, póngales nombres que digan algo (`df_limpio`, `gastos_2021`) en vez de `df1` y `df2`.

### El primer error que va a ver en su vida: `FileNotFoundError`

Vale más provocarlo ahora, con calma, que descubrirlo en el reto. La celda de abajo intenta leer el
archivo **sin** la parte `../datos/`, que es exactamente lo que la mitad del salón escribe la primera
vez. El `try / except` está para que el error se imprima sin interrumpir el notebook.

In [ ]:
try:
    pd.read_csv('estados_financieros.csv')
except FileNotFoundError as error:
    print("Tipo de error:", type(error).__name__)
    print("Mensaje:", error)

# Cuando le pase de verdad, la pregunta correcta es "¿desde dónde está mirando el notebook?":
import os
print()
print("El notebook está parado en:", os.getcwd())
print("Y el archivo lo busca ahí adentro, siguiendo la ruta que usted escriba.")

---

## 2.1 Si usted ya usó Excel, ya sabe la mitad de esto

Casi nadie llega aquí habiendo programado, y casi todo el mundo ha visto una hoja de cálculo. Esa
hoja es el mejor punto de partida que existe para pandas: **las piezas son las mismas, cambia quién
las mueve**. En Excel las mueve el ratón; aquí las mueve una línea escrita.

| En Excel | En pandas | Dónde aparece |
|----------|-----------|---------------|
| Una hoja de cálculo | Un **DataFrame** | Sección 2, el `df` que acaba de cargar |
| Un libro con varias hojas | Varios DataFrames a la vez, cada uno con su nombre de variable | Secciones 4 y 10, que crean tablas pequeñas al vuelo |
| Una columna (`A`, `B`, `C`) | Una **Series**, que se llama por su nombre y no por una letra | Sección 5 |
| El número de fila (1, 2, 3...) | El **índice** | Sección 3.1 |
| El formato de la columna (Número, Texto, Fecha) | El **dtype** | Sección 3.2 |
| Una celda vacía | `NaN` | Sección 3.2 |
| El embudo de "Filtrar" | La **máscara booleana** | Sección 7 |
| Chulear varios valores en la lista del filtro | `.isin([...])` | Sección 11 |
| Una tabla dinámica | `groupby` | Clase 4 |

Traer ese mapa puesto ahorra la mitad del camino. La otra mitad es saber **dónde la comparación deja
de servir**, y son tres puntos. Los tres se hacen visibles hoy mismo, así que vale la pena tenerlos
nombrados de antemano.

### Los tres puntos donde la analogía se rompe

**1. En Excel el tipo es de la celda; en pandas es de la columna.** Una hoja de cálculo acepta sin
protestar una columna con textos y números mezclados: cada celda decide por su cuenta. pandas no:
al leer el archivo le asigna **un solo tipo a la columna entera**, y si un valor no encaja, gana el
tipo más permisivo, que es texto. Consecuencia inmediata, que ya está esperándolo dos secciones más
abajo: `Valor (Miles)` viene con coma de miles, así que la columna completa quedó como texto y no
suma (sección 3.2; se arregla en la 4). Consecuencia silenciosa: preguntarle `== '2021'` a una
columna de números no da error, da cero filas. La sección 10.2 lo provoca a propósito.

**2. En Excel se edita una celda; en pandas se describe una transformación.** Quitarle la coma a
318 valores en Excel es corregirlos a mano, o escribir una fórmula en una columna nueva y arrastrarla
hacia abajo. Aquí es una línea que se aplica a los 318 de un golpe (sección 4) y que **queda
escrita**. Esa es la diferencia de fondo, y no es de comodidad: en la hoja queda el resultado, en el
cuaderno queda el procedimiento. Cualquiera puede volver a ejecutarlo el mes entrante sobre datos
nuevos y obtener lo mismo. Por eso lo que este curso recibe son notebooks y no archivos de datos: se
revisa cómo llegó al número, no el número.

**3. Excel muestra todo el tiempo lo que hay; pandas no muestra nada salvo que se lo pida.** Al
abrir una hoja usted ve la cuadrícula, y de reojo ya sabe si hay huecos o si una columna trae texto
donde debería haber cifras. `pd.read_csv` no imprimió ni una fila: dejó la tabla en memoria y se
quedó callado. Un dataset con problemas se ve exactamente igual que uno impecable, porque no se ve.
De ahí la regla de la sección siguiente: hay tres métodos que se ejecutan **siempre** sobre
cualquier dataset nuevo, antes de tocar nada. En Excel mirar es gratis; en pandas mirar es una
decisión, y el analista que no la toma trabaja a ciegas.

---

## 3. Los 3 grandes

Tres métodos que se ejecutan **siempre**, en este orden, sobre cualquier dataset nuevo. Responden
tres preguntas distintas: cómo se ven los datos, qué hay dentro, y cómo se distribuyen.

**Qué es un método.** Una función que le pertenece a un objeto y se llama con un punto:
`df.head()`. `head` no existe suelta; existe *dentro de* los DataFrames. Los paréntesis significan
"ejecútalo". Sin paréntesis, `df.head` no ejecuta nada: le devuelve la función misma.

### 3.1 `head()` — ¿cómo se ven los datos?

Muestra las primeras filas (5 por defecto, o las que le pida con `head(10)`). Es la primera mirada,
siempre.

Nunca escriba `print(df)` con el DataFrame completo: si tiene 20.000 filas, acaba de llenar la
pantalla de basura y de paso congelar el kernel. **Todo `df` en pantalla lleva `.head()`.**

In [ ]:
df.head()

**Qué es el índice.** La columna de la izquierda sin nombre no es un dato: es el **índice**
(*index*), la etiqueta con la que pandas identifica cada fila. Como no venía en el archivo, pandas
lo creó numerando desde 0.

Es una **etiqueta**, no una posición. Hoy coinciden porque la numeración es corrida, pero en la
clase 3, cuando borremos filas, van a dejar de coincidir: si borra la fila 2, la siguiente sigue
llamándose 3 aunque ahora ocupe el segundo lugar. Analogía: el índice es el número de cédula de la
fila. Si mueve la fila de lugar, la cédula la sigue.

### 3.2 `info()` — ¿qué hay dentro?

El resumen técnico. Hay que leer tres cosas explícitamente:

1. **Cuántas filas** hay en total (la línea `RangeIndex`).
2. **Qué tipo tiene cada columna** (la columna `Dtype`).
3. **Cuántos valores no nulos** tiene cada una (`Non-Null Count`). Si ese número es menor que el
   total de filas, hay datos faltantes.

**Qué es un tipo de dato (*dtype*).** La clase de valor que pandas cree que guarda una columna, y
que determina qué operaciones la columna acepta. Los que va a ver este semestre:

| Tipo | Qué es | Qué permite |
|------|--------|-------------|
| `int64` | Número entero | Sumar, promediar, comparar con `>` |
| `float64` | Número con decimales | Lo mismo, y admite `NaN` |
| `str` | Texto | Comparar con `==`, buscar fragmentos. **No** sumar |
| `bool` | Verdadero o falso | Es el tipo de las máscaras de la sección 7 |
| `datetime64` | Fecha y hora | Extraer año, mes, restar fechas. Clase 4 |

**Qué es `NaN`.** *Not a Number*: la marca que usa pandas para "aquí no hay valor". No es un cero
ni una cadena vacía: es la ausencia de dato. Aparece cuando el archivo trae una celda en blanco.
Este dataset no tiene ninguno, pero el del reto sí, y la clase 3 se dedica a ellos.

Una nota de versión: con pandas 3 las columnas de texto aparecen como `str`. En tutoriales más
viejos las verá como `object`. Es lo mismo, léalo como "texto".

In [ ]:
df.info()

**Pregunta de interpretación 1.** ¿Qué tipo tiene la columna `Valor (Miles)`? ¿Es el que usted
esperaría para una columna que contiene montos de dinero? Escriba por qué cree que pandas la leyó
así.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

`Valor (Miles)` es de tipo `str`, o sea texto. No es lo esperable para dinero: uno esperaría
`int64`. pandas la leyó como texto porque los valores traen coma como separador de miles
(`113,718,902`), y una coma dentro de un valor no forma parte de ningún número válido para pandas.
Ante la duda, pandas no adivina ni descarta: guarda el valor tal cual, como texto. La consecuencia
práctica es que ahora mismo no se puede sumar, promediar ni comparar esa columna con `>`.

</details>

### 3.3 `describe()` — ¿cómo se distribuyen los números?

`describe()` devuelve el resumen estadístico de las columnas **numéricas**: cuántos valores hay,
promedio, desviación estándar, mínimo, máximo y los percentiles 25, 50 y 75. Es la forma más rápida
de detectar un valor imposible (una edad de 200 años, un precio negativo).

Ejecútelo ahora, **antes** de arreglar nada, y mire bien qué devuelve. Lo que sale no es lo que
esperaría, y ese es justamente el punto.

In [ ]:
df.describe()

**Pregunta de interpretación 2.** `describe()` no devolvió promedio, ni mínimo, ni máximo.
Devolvió `count`, `unique`, `top` y `freq`. ¿Por qué? ¿Qué tendría que pasar para que devolviera un
promedio?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque en este DataFrame no hay **ninguna** columna numérica: las tres son texto. Cuando no
encuentra números, `describe()` cambia de repertorio y describe texto: cuántos valores hay
(`count`), cuántos distintos (`unique`), cuál es el más frecuente (`top`) y cuántas veces aparece
(`freq`). No está fallando, está describiendo otra cosa.

Para que devolviera un promedio tendría que haber al menos una columna de tipo numérico. Es lo que
vamos a hacer en la sección 4: los datos no van a cambiar, va a cambiar **cómo pandas los
entiende**. Esa es la mejor justificación que existe de por qué los tipos importan.

</details>

---

## 4. Arreglo express de los tipos

Este tramo es un **préstamo de la clase 3**. Lo hacemos hoy porque sin columnas numéricas no
podemos filtrar por valor, que es de lo que trata esta clase. En la clase 3 aprendemos a hacerlo
bien, con `pd.to_numeric()` y manejo de errores.

**Qué es `.str`.** El *accesorio de texto* de pandas. Una columna de texto no es un string de
Python: es una Series con 318 strings adentro. `.str` es la puerta que da acceso a las operaciones
de texto de siempre (`replace`, `lower`, `strip`, `contains`) pero aplicadas a los 318 valores **de
un solo golpe**, sin escribir un bucle. Esa es la idea que hace a pandas rápido de escribir: se
opera sobre columnas enteras, no sobre valores uno por uno.

Dos pasos por columna:

1. `.str.replace(',', '')` quita las comas del texto: `'113,718,902'` pasa a `'113718902'`.
2. `.astype(int)` convierte ese texto a número entero. `astype` es "trátame esta columna como si
   fuera de este otro tipo"; si algún valor no se puede convertir, revienta (y eso es bueno: le
   avisa en vez de inventar).

**Qué hace el signo `=` de la izquierda.** `df['Año'] = ...` **reemplaza** la columna dentro del
DataFrame por el resultado del lado derecho. Si el nombre no existiera, crearía una columna nueva.
Por eso esta celda se ejecuta **una sola vez**: si la ejecuta dos veces, la segunda intentará
aplicar `.str` a una columna que ya es numérica y le dará error. Si eso pasa, vuelva a ejecutar la
celda de `read_csv` y siga desde ahí.

In [ ]:
df['Año'] = df['Año'].str.replace(',', '').astype(int)
df['Valor (Miles)'] = df['Valor (Miles)'].str.replace(',', '').astype(int)

df.dtypes

### `describe()` otra vez, ahora con números de verdad

Ahora que hay dos columnas numéricas, `describe()` tiene algo que decir. Y de paso aparecen los
métodos de resumen que se usan todo el semestre: `.max()`, `.min()`, `.mean()` (promedio), `.sum()`
y `.count()`, todos aplicados **a la columna**, no al DataFrame completo.

In [ ]:
# Ahora describe() sí resume: las dos columnas numéricas tienen estadísticos.
print(df.describe())
print()

maximo_valor = df['Valor (Miles)'].max()
print("Valor máximo de la columna 'Valor (Miles)':", maximo_valor)
print("Valor mínimo:", df['Valor (Miles)'].min())
print("Cuántos valores hay:", df['Valor (Miles)'].count())

**Pregunta de interpretación 3.** El máximo de `Valor (Miles)` y el promedio que aparece en
`describe()` están muy lejos el uno del otro. ¿Qué le dice eso sobre cómo están repartidos los
valores de esta columna?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Que la columna está **muy desbalanceada**: hay unos pocos rubros enormes y muchísimos pequeños. Un
promedio que queda lejos del máximo, o lejos de la mediana, es la señal de que unos pocos valores
extremos están tirando de él.

Esa asimetría no es un error del archivo: es la naturaleza de un estado financiero, donde
"Ingresos Fiscales" y "Papelería" viven en la misma columna. Lo importante hoy es notar que **el
promedio solo no describe bien esta columna**. En la clase 4 esta observación se vuelve el tema
central: media contra mediana, y por qué reportar una sola de las dos puede ser engañoso.

</details>

**Pregunta de interpretación 4.** Mire el mínimo de `Valor (Miles)` en la salida de `describe()`.
¿Tiene sentido que haya valores negativos en un estado financiero?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Sí tiene sentido, y por eso es un buen ejemplo de por qué no se puede analizar un dataset sin saber
de qué habla. En contabilidad pública hay rubros que **restan**: devoluciones y descuentos,
reversiones de deterioro, ajustes de ejercicios anteriores. Se registran en negativo justamente
porque disminuyen el total.

La lección general: un valor raro no es automáticamente un error. Antes de "corregirlo" hay que
preguntarse si el dominio lo permite. Un salario negativo casi seguro es un error de captura; una
devolución negativa es exactamente lo que debe ser. En la clase 3, cuando hablemos de valores
atípicos (*outliers*), esta distinción va a ser el criterio central.

</details>

> **Para entender qué está pasando · pandas ignora los faltantes, y no avisa**
>
> En la sección 3 quedó dicho qué es `NaN`: la marca de "aquí no hay valor". Falta la consecuencia,
> que es la que hace daño. **Cuando usted calcula, pandas descarta los `NaN` en silencio.** No
> lanza un error, no imprime una advertencia: devuelve un número. Y ese número es el promedio de
> las filas que **sí** tenían dato, no el de todas.
>
> La celda de abajo lo provoca con cuatro valores, uno de ellos vacío. Compare lo que pandas
> devuelve con lo que saldría si el hueco contara como cero.
>
> Descartarlos no es un defecto: casi siempre es lo que uno quiere. Lo peligroso es no saberlo,
> porque entonces un promedio calculado sobre la mitad de las filas se lee como si fuera el de
> todas, y nada en la pantalla dice lo contrario. **La costumbre profesional: mire el `.count()` al
> lado del `.mean()`**, o `df.info()`, que trae los no nulos de cada columna. Si el conteo no llega
> al número de filas del DataFrame, el promedio habla de menos casos de los que usted cree.
>
> Este dataset no tiene ni un `NaN`, y por eso hay que fabricarlo para verlo. El del reto sí los
> tiene, y la clase 3 se dedica entera a ellos.

In [ ]:
# Cuatro sueldos, y uno que nunca se registró.
sueldos = pd.Series([1000, 2000, None, 5000])

print("Elementos en la Series:", len(sueldos))
print("Cuántos tienen valor: ", sueldos.count())
print()
print("sueldos.mean() ->", sueldos.mean(), " <- promedia 3 valores, no 4")
print("Si el hueco contara como 0 ->", sueldos.sum() / len(sueldos))
print()
print("Ni un error, ni una advertencia. Solo un número que responde otra pregunta.")

---

## 5. Seleccionar columnas: Series o DataFrame

**Qué es una Series.** Una sola columna, con su índice pegado. Una dimensión, no dos. Toda columna
que usted saque de un DataFrame es una Series; y un DataFrame, visto desde adentro, es un conjunto
de Series que comparten el mismo índice.

La regla que va a usar todo el semestre:

- **Un par de corchetes** devuelve una **Series** (una columna suelta, unidimensional).
- **Dos pares de corchetes** devuelven un **DataFrame** (una tabla, aunque tenga una sola columna).

Los corchetes internos no son decoración: son una **lista** de Python. `df[['CUENTA']]` es "dame
las columnas de esta lista", y una lista de un elemento sigue siendo una lista, así que el
resultado sigue siendo una tabla.

No lo memorice, compruébelo. `type()` es una función de Python que le dice qué clase de objeto
tiene en la mano.

> **Para entender qué está pasando · qué es una lista**
>
> Los corchetes internos de `df[['CUENTA']]` no son un invento de pandas: son una **lista de
> Python**. Conviene nombrarla, porque va a aparecer todo el semestre y nadie la presenta.
>
> Una lista es una **colección ordenada de valores**, escrita entre corchetes y separada por comas.
> `['Año', 'CUENTA']` es una lista de dos textos; `[2017, 2018, 2019]`, una de tres números. Puede
> tener un solo elemento (`['CUENTA']`) o ninguno (`[]`). Y **el orden importa**: es el orden en que
> salen las columnas.
>
> Una lista se puede guardar en una variable, y ahí es donde se gana legibilidad:
>
> ```python
> columnas = ['Año', 'CUENTA']
> df[columnas].head()
> ```
>
> Es exactamente lo mismo que escribir los corchetes adentro. Con dos columnas da igual; con ocho,
> la variable es la diferencia entre revisar la lista en un solo lugar o buscarla en medio de una
> línea kilométrica.
>
> Dónde más la va a ver hoy: `.isin(['Educación', 'Salud'])`, en la sección 11, recibe una lista, y
> `df.columns.tolist()`, en la sección 9, **devuelve** una.

In [ ]:
una_columna = df['CUENTA']
tabla_de_una_columna = df[['CUENTA']]

print("df['CUENTA']    ->", type(una_columna))
print("df[['CUENTA']]  ->", type(tabla_de_una_columna))

**Por qué importa tanto.** Casi todo error raro de principiante en pandas es "creí que tenía un
DataFrame y tenía una Series", o al revés. Los dos objetos se imprimen parecido pero no aceptan los
mismos métodos: `.shape` de una Series devuelve un solo número, no una pareja; y un DataFrame no
tiene `.str`. Cuando un método le diga que no existe, la primera pregunta es **qué objeto tengo en
la mano**, y se responde con `type()`.

Para seleccionar varias columnas, la lista va dentro de los corchetes externos. El orden de la
lista es el orden en que salen las columnas.

In [ ]:
df[['Año', 'CUENTA']].head()

### Seleccionar varias columnas a la vez

Dos pares de corchetes, una lista de nombres dentro, y el orden de esa lista manda: el resultado
sale en el orden que usted pidió, no en el que tenía el archivo. Y como se pidieron dos columnas,
lo que vuelve es un DataFrame, no una Series.

In [ ]:
tabla_ocho = df[['CUENTA', 'Valor (Miles)']].head(8)

print("Tipo del resultado:", type(tabla_ocho).__name__)
print("Forma:", tabla_ocho.shape)
print()
print(tabla_ocho)

**Pregunta de interpretación 5.** La celda de arriba pidió dos columnas y devolvió un DataFrame. Si
se hubiera pedido una sola, con un par de corchetes, ¿qué habría devuelto y por qué importa la
diferencia?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Habría devuelto una **Series**: una sola columna con su índice pegado. Importa porque muchas cosas
que funcionan sobre un DataFrame no funcionan igual sobre una Series, y al revés. El error de
principiante más común del semestre es creer que se tiene un DataFrame cuando se tiene una Series.

La regla operativa: **un par de corchetes devuelve una Series, dos pares devuelven un DataFrame.**
Y la forma de salir de la duda no es acordarse, es preguntar: `type(lo_que_sea)`.

</details>

---

## 6. Seleccionar filas por posición: `.iloc`

Hasta aquí seleccionamos columnas. Para filas hay dos herramientas:

- **`.iloc`** selecciona por **posición** (la *i* es de *integer location*): `df.iloc[0]` es la
  primera fila, `df.iloc[0:5]` son las cinco primeras. Como en cualquier lista de Python, se cuenta
  desde 0 y el final del rango no se incluye.
- **`.loc`** selecciona por **etiqueta del índice**. Hoy dan lo mismo porque el índice es 0, 1, 2...
  y coincide con la posición. Dejarán de coincidir en la clase 3, y ahí `.loc` se vuelve
  importante. También reaparece al final de la sección 10.

Esto sirve para **mirar**, no para analizar. Filtrar por posición casi nunca responde una pregunta
de negocio: nadie pregunta "¿qué dice la fila 47?". Preguntan "¿qué rubros pasaron de mil
millones?". Para eso está la sección siguiente, que es el corazón de la clase.

Fíjese en el detalle: una fila suelta sale como **Series** (los nombres de columna pasan a ser el
índice), y un rango de filas sale como **DataFrame**.

In [ ]:
print("Una fila (Series):")
print(df.iloc[0])
print()
print("Cinco filas (DataFrame):")
print(df.iloc[0:5])

---

## 7. La máscara booleana, en dos pasos

Aquí empieza lo importante. Todo lo demás de la clase es andamiaje para esto.

**La analogía de pasar lista.** El profesor recorre la lista del curso y, al frente de cada nombre,
escribe una sola marca: vino o no vino. No hay "más o menos" y no se salta a nadie. Cuando termina,
lo que tiene en la mano **no es el grupo de los que vinieron**: es la **planilla marcada**, con
exactamente tantas marcas como nombres tiene la lista. Los que vinieron aparecen después, cuando
alguien lee la planilla y se queda con los que tienen marca de "sí".

Ahí está el paso que cuesta ver, y es el único difícil de hoy: **primero se marca la lista completa,
y solo después se usa la marca para escoger.** Son dos cosas distintas, en dos momentos distintos.
La planilla se puede mirar, contar y corregir antes de que nadie se quede afuera.

**Qué es una máscara booleana.** Exactamente esa planilla: una Series de valores `True` y `False`,
**del mismo largo** que el DataFrame, donde cada elemento es la marca que le quedó a una fila
después de hacerle la pregunta a una columna. "Booleana" viene de *booleano*, el tipo de dato que solo puede valer
verdadero o falso.

**Filtrar es dos pasos, no uno:**

1. **Construir la máscara**: hacerle una pregunta a una columna. Devuelve la Series de `True`/`False`.
2. **Aplicar la máscara**: pasarla entre corchetes al DataFrame. Devuelve solo las filas donde la
   máscara dijo `True`.

Ejercicio mental antes de ejecutar nada: si tuviera cinco filas con los valores 10, 40, 25, 80, 5 y
preguntara "¿es mayor que 30?", la máscara sería `False, True, False, True, False`, y al aplicarla
quedarían dos filas. Si eso le quedó claro, ya entendió *boolean indexing*.

Vamos a hacerlo lento, paso por paso. Después lo comprimimos.

### Paso 1 · Construir la máscara y guardarla en una variable

In [ ]:
mascara = df['Valor (Miles)'] > 1000000

mascara.head(10)

Eso es la planilla marcada. Una marca por fila, en el mismo orden que el DataFrame y con el mismo
índice.

**Fíjese en lo que no pasó:** `df` no cambió. Construir una máscara no filtra nada; solo produce
las marcas.

### Paso 2 · Verificar que la máscara tiene el largo correcto

La verificación más barata que existe: la máscara siempre tiene exactamente tantos elementos como
filas tiene el DataFrame. Si no coinciden, algo está mal (lo más común: construyó la máscara sobre
un DataFrame y la está aplicando a otro).

In [ ]:
print("Largo de la máscara:", len(mascara))
print("Filas del DataFrame:", len(df))
print("¿Coinciden?", len(mascara) == len(df))

### Paso 3 · Contar cuántos `True` hay

En pandas, `True` vale 1 y `False` vale 0. Entonces `mascara.sum()` **cuenta** cuántas filas cumplen
la condición, y lo hace **antes** de aplicarla. Este truco lo va a usar todo el semestre: cuando una
pregunta sea "¿cuántos...?", la respuesta es una suma de máscara, no un DataFrame filtrado.

`mascara.mean()` es el mismo truco un paso más allá: el promedio de unos y ceros es la
**proporción** de filas que cumplen. Un número entre 0 y 1.

In [ ]:
print("Filas que cumplen la condición:", mascara.sum())
print("De un total de:", len(df))
print("Proporción:", round(mascara.mean(), 3))

### Paso 4 · Aplicar la máscara

Recién ahora se filtra. Y una advertencia que evita horas de confusión: **filtrar no modifica `df`**.
`df[mascara]` construye una tabla nueva e independiente y se la devuelve. Si no la guarda en una
variable, se imprime y se pierde.

In [ ]:
valores_altos = df[mascara]

print("Forma del resultado:", valores_altos.shape)
print("¿df cambió?", df.shape)
valores_altos.head()

### La forma comprimida

Una vez que entendió los cuatro pasos, se puede escribir todo en una línea. Es exactamente lo
mismo: la máscara se construye adentro de los corchetes en vez de guardarse en una variable. Esta
es la forma que va a ver en todo el código del mundo, y la que se lee raro al principio porque
`df` aparece dos veces. Léala de adentro hacia afuera: primero la pregunta, después el filtro.

**Regla práctica:** cuando algo no le funcione, vuelva a partirlo en dos pasos y mire la máscara.
El 90% de los errores de filtrado se ven a simple vista en la máscara.

### Los operadores de comparación

| Operador | Significado | Ejemplo |
|----------|-------------|---------|
| `==` | igual a | `df['Año'] == 2021` |
| `!=` | distinto de | `df['Año'] != 2021` |
| `>` | mayor que | `df['Valor (Miles)'] > 0` |
| `<` | menor que | `df['Valor (Miles)'] < 0` |
| `>=` | mayor o igual | `df['Año'] >= 2019` |
| `<=` | menor o igual | `df['Año'] <= 2018` |

Ojo con `==`: **dos** signos igual. Uno solo (`=`) es asignación, no comparación. `x = 5` significa
"guarda 5 en x"; `x == 5` pregunta "¿x vale 5?".

In [ ]:
print(df[df['Valor (Miles)'] > 1000000].shape)

### La máscara, otra vez y en dos pasos separados

Aquí se ve el punto entero de la analogía de pasar lista: **primero se marca la planilla completa,
después se aplica**. La máscara se guarda en su propia variable a propósito, para poder mirarla.

In [ ]:
# Paso 1: marcar la planilla completa. NO se aplica todavía.
mascara_2021 = df['Año'] == 2021

print("Largo de la máscara:", len(mascara_2021), " <- igual al número de filas del DataFrame")
print("Cuántas filas cumplen:", mascara_2021.sum())
print()

# Paso 2: aplicarla.
print(df[mascara_2021].head())

**Pregunta de interpretación 6.** La máscara tiene 318 elementos, pero al aplicarla quedan muchas
menos filas. ¿Por qué la máscara mide 318 y no lo que mide el resultado?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Porque la máscara es **una marca por fila**, no la lista de las filas que pasaron. Se le preguntó a
las 318 filas "¿es de 2021?" y quedaron anotadas 318 marcas: unas `True` y otras `False`. Ese es el
tamaño de la planilla. El filtrado viene después, y es el que se queda solo con las `True`.

De ahí sale la verificación más barata que existe: **si el largo de la máscara no coincide con el
número de filas del DataFrame, algo está mal** —lo más común es que se aplicó antes de guardarla, y
entonces se está filtrando dos veces.

</details>

---

## 8. Combinar condiciones: `&`, `|`, `~`

Dos planillas del mismo curso: una marca quién asistió, la otra marca quién entregó el taller.
Combinarlas es leer las dos marcas de cada estudiante y decidir si hace falta que ambas digan sí, o
si basta con que una lo diga.

| Operador | Significado | Se lee | Se escribe con |
|----------|-------------|--------|----------------|
| `&` | AND | ambas condiciones deben ser verdaderas | Shift + 6 (o el símbolo `&`) |
| `\|` | OR | basta con que una sea verdadera | la barra vertical, Alt Gr + 1 en teclado español |
| `~` | NOT | invierte la máscara: los `True` pasan a `False` y viceversa | Alt Gr + 4, o Alt + 126 |

Estos operadores trabajan **elemento por elemento**: comparan la decisión 1 de una máscara con la
decisión 1 de la otra, la 2 con la 2, y así hasta el final. Por eso las dos máscaras tienen que
tener el mismo largo.

**Regla no negociable: cada condición va entre paréntesis.** Siempre. En la sección 9 va a ver, con
sus ojos, qué pasa si se olvidan.

### `&` — las dos cosas a la vez

In [ ]:
# Rubros del año 2021 con valor mayor a un millón (de miles de pesos)
resultado = df[(df['Año'] == 2021) & (df['Valor (Miles)'] > 1000000)]

print("Filas:", resultado.shape[0])
resultado.head()

### `|` — cualquiera de las dos

Cuidado con el lenguaje corriente: cuando alguien dice "quiero los datos de 2016 **y** de 2021",
está pidiendo un `|`, no un `&`. Ninguna fila puede ser de 2016 **y** de 2021 a la vez, así que un
`&` ahí devuelve cero filas. Traducir del español al operador correcto es la mitad del trabajo de
esta clase, y es exactamente lo que pide el reto.

In [ ]:
# Rubros del primer año o del último
resultado = df[(df['Año'] == 2016) | (df['Año'] == 2021)]

print("Filas:", resultado.shape[0])
print("Años presentes en el resultado:", sorted(resultado['Año'].unique().tolist()))

### `~` — todo lo que no

In [ ]:
# Todo lo que NO es del año 2016
sin_2016 = df[~(df['Año'] == 2016)]

print("Con ~ :", sin_2016.shape[0], "filas")
print("Con !=:", df[df['Año'] != 2016].shape[0], "filas")

`~(col == valor)` y `col != valor` dan lo mismo cuando hay una sola condición, y en ese caso `!=`
se lee mejor. `~` se vuelve imprescindible cuando lo que quiere negar es una condición
**compuesta**: `~((df['Año'] == 2016) & (df['Valor (Miles)'] > 0))` es "todo lo que no sea, a la
vez, de 2016 y positivo". Escribir eso con `!=` es un ejercicio de lógica innecesario.

### Tres condiciones a la vez, y el agrupamiento que decide el resultado

Años 2019 **o** 2020, y además valor **negativo**. Son tres condiciones: las dos de año van unidas
con `|` dentro de su propio paréntesis, y esa combinación se une con `&` a la del valor. Fíjese en
el paréntesis grande que envuelve el `|`: sin él, el resultado es otro.

In [ ]:
negativos_1920 = df[((df['Año'] == 2019) | (df['Año'] == 2020)) & (df['Valor (Miles)'] < 0)]

print("Filas que cumplen las tres condiciones:", len(negativos_1920))
print()
print(negativos_1920)

**Pregunta de interpretación 7.** Si se quitara el paréntesis grande que envuelve las dos
condiciones de año, ¿qué pregunta estaría haciendo el código en vez de la que se quería hacer?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Estaría preguntando "**año 2019, o bien (año 2020 y valor negativo)**", que es otra pregunta: dejaría
entrar todas las filas de 2019 aunque su valor fuera positivo. El `&` se evalúa antes que el `|`, así
que sin el paréntesis grande Python agrupa por su cuenta y agrupa distinto de como usted lo leyó.

Por eso la regla del curso no es aprenderse la tabla de precedencia: es **poner paréntesis siempre**,
en cada condición y alrededor de cada grupo. Cuesta cuatro caracteres y elimina la clase entera de
errores que no revientan pero devuelven la respuesta equivocada.

</details>

---

## 9. Las tres trampas que sí avisan

Estas fallan con un error en rojo. Son las buenas: el error es el mensaje. Las de la sección 10 son
peores, porque no dicen nada.

Cada celda de abajo provoca el error a propósito y lo atrapa con `try / except` para que el
notebook siga corriendo. Lea el nombre del error y el mensaje: son los que va a ver en el reto.

### Trampa 1 · Paréntesis olvidados

Sin paréntesis, Python agrupa mal la expresión. `&` tiene **más precedencia** que `==`, así que
Python intenta resolver primero lo que está pegado al `&`, en el medio de la expresión, y desde ahí
todo se descarrila.

Lo interesante es que el error que sale **no siempre es el mismo**: depende de con qué tipos se
tropiece Python al agrupar mal. Los dos casos de abajo son el mismo olvido y producen errores
distintos. No hay que memorizar cuál sale cuándo; hay que reconocer que ambos significan lo mismo:
**faltan paréntesis**.

In [ ]:
# Caso A: comparando contra un texto
try:
    df[df['CUENTA'] == 'Salud' & df['Año'] > 2019]
except Exception as error:
    print("Caso A ->", type(error).__name__)
    print("  ", error)

# Caso B: comparando contra números
try:
    df[df['Año'] == 2021 & df['Valor (Miles)'] > 1000000]
except Exception as error:
    print("Caso B ->", type(error).__name__)
    print("  ", error)

# La versión correcta, para comparar:
print()
print("Con paréntesis:", df[(df['CUENTA'] == 'Salud') & (df['Año'] > 2019)].shape[0], "filas")

No es una convención de estilo: sin paréntesis, el código **no corre**. Y note que en el caso B
el error habla de "el valor de verdad de una Series", que es literalmente el mensaje de la trampa
2. Por eso conviene tener el reflejo: cuando aparezca ese mensaje, revise **primero** los
paréntesis y **después** los operadores.

### Trampa 2 · `and` en vez de `&`

En Python normal se escribe `and` y `or`. Con columnas de pandas **no funcionan**.

La razón en español: `and` espera **una** respuesta sí/no y devuelve una. Usted le está pasando 318
respuestas de golpe. Python no sabe si "esta Series es verdadera" significa que todas lo son, que
alguna lo es, o que no está vacía. Prefiere lanzar un error antes que adivinar.

`&` sí está definido por pandas para operar elemento por elemento, y por eso es el que sirve.

In [ ]:
try:
    df[(df['Año'] == 2021) and (df['Valor (Miles)'] > 1000000)]
except Exception as error:
    print(type(error).__name__)
    print(error)

### Trampa 3 · El nombre de la columna mal escrito

Los nombres de columna **distinguen mayúsculas** y respetan tildes y espacios. `df['año']` no es
`df['Año']`, y `df['Valor(Miles)']` no es `df['Valor (Miles)']`. El error se llama `KeyError`, y la
"llave" que aparece entre comillas es lo que usted escribió, no lo que existe.

La cura no es adivinar: es pedir la lista real y copiar y pegar.

In [ ]:
try:
    df['año']
except KeyError as error:
    print(type(error).__name__, "->", error)

print()
print("Los nombres reales, para copiar y pegar:")
print(df.columns.tolist())

---

## 10. Los cuatro errores que no avisan

Estos son los peligrosos. El código corre, no sale nada en rojo, y el resultado está mal. Si algún
día su análisis dice una barbaridad y no encuentra por qué, empiece por aquí.

### 10.1 El filtro devuelve 0 filas y el código se ve bien

Casi siempre no es el código: es un supuesto equivocado sobre cómo están escritos los datos.
Mayúsculas, tildes, espacios sobrantes, o un valor que sencillamente no existe.

**El reflejo correcto: antes de dudar del código, dude del dato.** `df['columna'].unique()` le
muestra los valores que realmente hay.

In [ ]:
# Un filtro perfectamente escrito... que devuelve nada
print("Buscando 'total ingresos':", df[df['CUENTA'] == 'total ingresos'].shape[0], "filas")
print("Buscando 'Total Ingresos':", df[df['CUENTA'] == 'Total Ingresos'].shape[0], "filas")

# Cómo se averigua, en vez de adivinar:
print()
print("Valores que contienen 'ingreso', escritos como están en el dato:")
for valor in sorted(df['CUENTA'].unique()):
    if 'ngreso' in valor:
        print("  ", repr(valor))

En el reto va a pasar exactamente esto: el dataset de accidentes está **todo en mayúsculas y sin
tildes** (`MOTOCICLETA`, `ANTIOQUIA`, `CON MUERTOS`). Filtrar por `'Motocicleta'` devuelve cero
filas y el código se ve impecable. Ya sabe qué hacer: `.unique()`.

### 10.2 Comparar contra el tipo equivocado

Hermano gemelo del anterior, y todavía más difícil de ver, porque aquí el valor sí existe: lo que
no coincide es el **tipo**.

`df['Año']` es de tipo `int64` desde que la convertimos en la sección 4. Si usted la compara contra
`'2021'` **entre comillas**, le está preguntando a una columna de números si alguno de ellos es
igual a un texto. La respuesta honesta es que no, ninguno: pandas devuelve la máscara toda en
`False`, sin una sola queja. Cero filas, código impecable, valor que sí existe.

La regla es corta: **las comillas no son decoración, son el tipo del valor.** Si la columna es
numérica, el valor va sin comillas; si es de texto, con comillas. `df.dtypes` le dice cuál es cuál
en una línea.

El detalle curioso, que conviene ver: con `==` el error es silencioso, pero con `>` o `<` pandas sí
levanta un `TypeError`. No es incoherencia. Preguntar si un número es *igual* a un texto tiene una
respuesta sensata —no lo es—, mientras que preguntar si un número es *mayor* que un texto no
significa nada, y ahí pandas prefiere gritar antes que inventarse un orden.

In [ ]:
print("Tipo de la columna 'Año':", df['Año'].dtype)
print()

# Con comillas: le pregunta a una columna de numeros si es igual a un texto.
print("df['Año'] == '2021' ->", (df['Año'] == '2021').sum(), "filas. Ni un error, ni un aviso.")

# Sin comillas: la comparacion que si tiene sentido.
print("df['Año'] ==  2021  ->", (df['Año'] == 2021).sum(), "filas.")

# Y con un operador de orden, el mismo descuido si avisa:
print()
try:
    df['Año'] > '2020'
except TypeError as error:
    print("df['Año'] > '2020' ->", type(error).__name__)
    print("  ", error)

### 10.3 Buscar texto cuando hay valores faltantes

`.str.contains('MOTO')` busca un fragmento dentro del texto de cada fila. La pregunta incómoda es
qué debe responder cuando la fila **no tiene texto**, sino `NaN`. ¿Contiene 'MOTO' un dato que no
existe? Ni sí ni no.

Con pandas 3 y una columna leída de un CSV, pandas resuelve la duda por usted y responde `False`.
Pero con columnas de tipo `object` (datos armados a mano, o archivos leídos con versiones más
viejas) la máscara sale con huecos, y una máscara con huecos **no sirve para filtrar**: revienta.

Por eso la costumbre profesional es escribir siempre `na=False`, que significa "los faltantes
cuentan como no". No es un parche contra un error: es **decir explícitamente qué debe pasar con lo
que falta**, en vez de dejarlo al criterio de la versión que tenga instalada.

> **Para entender qué está pasando · qué es un parámetro con nombre**
>
> `na=False` no es un símbolo mágico ni parte del nombre del método: es un **parámetro con
> nombre**. Un método recibe lo que necesita de dos formas:
>
> - **Por posición**: `replace(',', '')`. Cuál es cuál lo decide el **orden**: el primero es lo que
>   se busca y el segundo por qué se reemplaza. Cambiarlos de lugar cambia lo que hace.
> - **Por nombre**: `contains('MOTO', na=False)`. Usted escribe `nombre=valor`, dice explícitamente
>   cuál está pasando, y el orden deja de importar.
>
> Los parámetros con nombre casi siempre traen un **valor por defecto**: si usted no los escribe, el
> método usa el suyo. Por eso `df.head()` funciona sin nada entre paréntesis y `df.head(8)` también.
> Escribirlos es tomar la decisión en vez de heredarla, y esa es toda la diferencia entre un
> resultado que usted eligió y uno que le tocó.
>
> En la celda de abajo hay dos: `pd.Series([...], dtype=object)` decide de qué tipo se construye la
> columna, y `contains('MOTO', na=False)` decide qué se responde donde no hay dato. Le quedan dos
> más por ver antes de terminar: `between(0, 1000, inclusive='left')` en la sección 11 y
> `to_string(index=False)` en la 12.
>
> Cómo se averigua cuáles acepta un método, sin buscar en internet: escriba `df.head?` en una celda
> y ejecútela. Jupyter abre la documentación con la lista de parámetros y sus valores por defecto.

In [ ]:
# Una columna con un hueco, del tipo genérico 'object'
ejemplo = pd.Series(['MOTOCICLETA', None, 'AUTOMOVIL'], dtype=object)

mascara_texto = ejemplo.str.contains('MOTO')
print("La máscara sale con un hueco:", mascara_texto.tolist())

try:
    ejemplo[mascara_texto]
except Exception as error:
    print("Al filtrar con ella ->", type(error).__name__)
    print("  ", error)

print()
print("Con na=False:", ejemplo[ejemplo.str.contains('MOTO', na=False)].tolist())

### 10.4 Modificar un filtro creyendo que modifica el original

Este es el más silencioso de todos, y el que más tiempo hace perder.

Ya lo dijimos en la sección 7: `df[mascara]` devuelve una **tabla nueva e independiente**. Si usted
guarda esa tabla y le cambia un valor, está cambiando la tabla nueva. El original ni se entera. No
hay error, no hay advertencia: el síntoma es el silencio, y por eso cuesta verlo.

Si lo que quiere es modificar el original, la herramienta es `.loc`, con la condición y la columna
en el mismo par de corchetes: `df.loc[condicion, 'columna'] = valor`. Se lee como una frase: "en
las filas que cumplen esto, en esta columna, pon este valor".

La celda de abajo trabaja sobre una **copia** del DataFrame (`df.copy()`) para no dañar el `df` que
venimos usando. Fíjese en los números, no en el código.

In [ ]:
ensayo = df.copy()

# Intento 1: modificar el resultado de un filtro. No toca el original.
solo_2021 = ensayo[ensayo['Año'] == 2021]
solo_2021['Valor (Miles)'] = 0
print("Después de 'modificar' el filtro, el máximo de 2021 en el original es:",
      ensayo[ensayo['Año'] == 2021]['Valor (Miles)'].max())
print("Ni un error, ni una advertencia. Y el original está intacto.")

# Intento 2: modificar el original de verdad, con .loc
ensayo.loc[ensayo['Año'] == 2021, 'Valor (Miles)'] = 0
print()
print("Con .loc, el máximo de 2021 en el original es:",
      ensayo[ensayo['Año'] == 2021]['Valor (Miles)'].max())
print("df original, sin tocar:", df[df['Año'] == 2021]['Valor (Miles)'].max())

---

## 11. Dos atajos que hacen el código legible

Todo lo que sigue se puede escribir con lo que ya sabe. Se usa igual, porque el código se escribe
una vez y se lee muchas.

### `.isin()` — cuando la cadena de `|` se vuelve ridícula

`df['col'].isin([lista])` construye la máscara "¿el valor de esta fila está en esta lista?".

In [ ]:
# Forma larga: tres condiciones unidas por |
largo = df[(df['CUENTA'] == 'Educación') | (df['CUENTA'] == 'Salud') | (df['CUENTA'] == 'Vivienda')]

# Forma corta: una lista
corto = df[df['CUENTA'].isin(['Educación', 'Salud', 'Vivienda'])]

print("Forma larga:", largo.shape[0], "filas")
print("Forma corta:", corto.shape[0], "filas")

Mismo resultado. La diferencia no es velocidad (a esta escala no se nota): es legibilidad. Cinco
condiciones unidas por `|` son ilegibles, y cada una es una oportunidad de escribir mal un nombre
de columna. `.isin([...])` es una sola lista, fácil de revisar y fácil de alargar.

### `.between()` — cuando lo que quiere es un rango

`df['col'].between(a, b)` es lo mismo que `(df['col'] >= a) & (df['col'] <= b)`. Es **inclusivo**
en ambos extremos por defecto: `between(0, 1000)` incluye el 0 y el 1000. Si necesita otra cosa,
existe el parámetro `inclusive='left'`, `'right'` o `'neither'`.

In [ ]:
# Rubros con valores entre 0 y 1000 (miles de pesos)
pequenos = df[df['Valor (Miles)'].between(0, 1000)]

print("Filas:", pequenos.shape[0])
pequenos.head()

### Los dos atajos juntos

La misma pregunta de antes, escrita para que se lea. `.isin()` reemplaza la cadena de `|` y
`.between()` reemplaza las dos comparaciones del rango. Nótese que `.between()` **incluye los dos
extremos**.

In [ ]:
medianos = df[df['Año'].isin([2017, 2018, 2019])
              & df['Valor (Miles)'].between(1000000, 10000000)]

print("Filas que cumplen:", len(medianos))
print()
print(medianos.head(10))

**Pregunta de interpretación 8.** La misma consulta se podía escribir con dos `|` y dos
comparaciones de rango. ¿Qué se gana usando `.isin()` y `.between()`, si el resultado es idéntico?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Se gana **legibilidad**, que en analítica no es un lujo: es lo que permite que alguien más audite lo
que usted hizo. Una línea con cuatro `|` encadenados y dos comparaciones se lee mal y esconde los
errores de paréntesis; `.isin([...])` y `.between(a, b)` dicen en voz alta cuál era la intención.

Y hay un segundo beneficio, menos obvio: cuando la lista de años crezca a diez, `.isin()` no cambia
de forma, mientras que la cadena de `|` se vuelve inmanejable y es donde se cuela el error.

</details>

---

## 12. Cerrar con una pregunta de verdad

Hasta ahora filtramos por filtrar. Los ejercicios eran de técnica. Pero un filtro no es el
entregable de nadie: es el instrumento. Lo que se entrega es una respuesta.

Una pregunta que un contador de esta entidad haría de verdad:

**¿Cómo evolucionó el total de ingresos entre 2016 y 2021?**

Se responde con lo que ya sabe: quedarse con las filas cuya `CUENTA` sea `'Total Ingresos'`,
mostrar solo las columnas que importan, y ordenar por año para poder leer la serie. `.sort_values()`
ordena por la columna que le indique, y `.to_string(index=False)` imprime la tabla sin el índice,
que aquí no aporta nada.

In [ ]:
total_ingresos = df[df['CUENTA'] == 'Total Ingresos'][['Año', 'Valor (Miles)']]

print(total_ingresos.sort_values('Año').to_string(index=False))

**Pregunta de interpretación 9.** ¿Los ingresos crecieron, cayeron o se mantuvieron? ¿Hay algún
año que se salga de la tendencia? Escriba una conclusión de dos frases, como se la diría al gerente
de la entidad.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Los ingresos crecen todos los años, sin excepción: pasan de unos 149.000 millones en 2016 a unos
275.000 millones en 2021, casi el doble en seis años. Lo que cambia es el **ritmo**: los saltos de
2016 a 2018 son grandes, 2019 y 2020 crecen mucho menos, y 2021 vuelve a acelerar con el mayor
salto de toda la serie.

Una conclusión defendible para el gerente: "los ingresos crecieron de forma sostenida durante todo
el periodo, pero el crecimiento se aplanó en 2019 y 2020 y se recuperó con fuerza en 2021". Note
que esto es una **descripción**, no una explicación: para decir *por qué* se aplanó habría que
mirar qué rubros lo empujaron, y eso es GroupBy, que es la clase 4.

Y una advertencia de honestidad que vale para todo el semestre: estas cifras están sin ajustar por
inflación. Parte de ese crecimiento es precios, no actividad. Decirlo en voz alta es lo que separa
un análisis de una tabla bonita.

</details>

---

## 13. Punto de control final

Este cuaderno no se autocalifica: no hay nada que teclear en él. El punto de control es usted
respondiéndose, sin abrir los desplegables, estas tres preguntas:

1. ¿Sabría decir, mirando una línea, si devuelve una Series o un DataFrame?
2. ¿Sabría construir una máscara, mirarla antes de aplicarla, y explicar por qué mide lo que mide?
3. ¿Sabría combinar tres condiciones con los paréntesis en su sitio?

Si alguna respuesta es "no", no pase de largo: la clase 3 arranca dando por sabido todo lo de este
cuaderno, y el reto del bloque 3 es donde sí hay que escribir. Levante la mano ahora, que el
profesor está en el salón.

---

## 14. Preguntas que siempre salen

**¿Puedo usar `.query()` en vez de todo esto?** Sí, y lo vemos en la clase 3. `.query("Año == 2021")`
escribe el filtro como una frase, sin corchetes ni paréntesis. Se aprende después a propósito:
`.query()` construye la máscara por debajo, así que quien lo aprende primero no sabe depurar cuando
falla.

**¿Cuántas filas aguanta pandas?** En un portátil normal, unos pocos millones sin quejarse. Las 318
de hoy y las 20.000 del reto no son "muchos datos" para pandas; para Excel, las 20.000 ya empiezan a
serlo. Cuando el archivo no quepa en memoria, las salidas son las de la sección 2: `nrows`,
`usecols`, `chunksize`, y más adelante otras herramientas.

**¿Y si me equivoco y daño los datos?** No puede. `read_csv` copia el archivo a memoria y nunca lo
vuelve a tocar. Lo peor que puede pasar es que deje el DataFrame hecho un desastre, y eso se arregla
volviendo a ejecutar la celda de `read_csv` desde arriba. Esa es una de las razones por las que un
notebook tiene que correr entero de arriba a abajo: es su botón de deshacer.

**¿Este dataset sirve para el proyecto de mi equipo?** No. Los CSV de las clases son material de
enseñanza, elegidos por lo que permiten enseñar, y varios ni siquiera llegan a los umbrales del
proyecto. El dataset del proyecto lo consigue cada equipo, de la fuente que quiera, y tiene que
poder decir de dónde salió y bajo qué condiciones lo usa. Los criterios están en la guía de entrega
del Momento 1.

**¿Por qué tanto español en un curso de programación?** Porque lo que se evalúa no es que el filtro
corra: es que usted sepa decir qué significa el número que salió. El código es el instrumento; la
frase es el entregable.

---

## Resumen

| Lo que hizo | Con qué |
|-------------|---------|
| Cargar un CSV | `pd.read_csv('../datos/estados_financieros.csv')` |
| Ver la forma | `df.shape`, `df.shape[0]`, `df.shape[1]` |
| Primera mirada | `df.head(n)` |
| Tipos y nulos | `df.info()`, `df.dtypes` |
| Estadísticas | `df.describe()` |
| Una columna (Series) | `df['col']` |
| Varias columnas (DataFrame) | `df[['col1', 'col2']]` |
| Filas por posición | `df.iloc[0]`, `df.iloc[0:5]` |
| Construir máscara | `mascara = df['col'] > valor` |
| Contar antes de filtrar | `mascara.sum()` |
| Aplicar máscara | `df[mascara]` |
| Combinar | `df[(cond1) & (cond2)]`, `|`, `~` |
| Lista de valores | `df['col'].isin([...])` |
| Rango | `df['col'].between(a, b)` |
| Modificar el original | `df.loc[condicion, 'col'] = valor` |
| Ver los valores reales | `df['col'].unique()` |

**Las cuatro reglas que no se negocian:**

1. Cada condición entre paréntesis.
2. `&` `|` `~`, nunca `and` `or` `not`.
3. Si el filtro devuelve 0 filas, imprima `df['col'].unique()` antes de dudar del código.
4. Filtrar nunca modifica el original. Si quiere modificarlo, `.loc`.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, está listo para el reto:

- [ ] Puedo cargar un CSV y decir cuántas filas y columnas tiene.
- [ ] Sé cuándo tengo una Series y cuándo un DataFrame, y sé cómo comprobarlo.
- [ ] Puedo construir una máscara, mirarla, contarla y aplicarla, en pasos separados.
- [ ] Puedo combinar dos condiciones sin que reviente, y sé qué significa el error si revienta.
- [ ] Sé qué hacer cuando un filtro me devuelve 0 filas.

**Lo que queda pendiente para la clase 3:** `.str.contains()` para filtrar texto por fragmento
en serio, `.query()` como sintaxis alternativa, y todo el arreglo formal de tipos, nulos,
duplicados y textos inconsistentes.

**Ahora:** el reto. `../reto/reto_starter.ipynb`, con 20.000 filas de accidentes de tránsito. Misma
técnica, datos que no vio aquí.